In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')
#os.chdir("/content/drive/MyDrive")

Mounted at /content/drive


In [3]:
#Load the CSV File

df = pd.read_csv("/content/drive/MyDrive/VAT datasets/ACCdata/accelerometer_2026-03-14_sitting.csv")

print(df.head())

       time     gFx     gFy     gFz   TgF
0  0.003195  0.0391  0.3577  0.8883  0.96
1  0.038135  0.0391  0.3567  0.8844  0.95
2  0.043385  0.0264  0.3547  0.9039  0.97
3  0.048266  0.0332  0.3508  0.9166  0.98
4  0.073928  0.0362  0.3577  0.9205  0.99


In [4]:
#Remove Metadata Rows (if present)

#Sometimes the file contains header information before the data.

df = df[['time','gFx','gFy','gFz','TgF']]
df = df.dropna()

In [5]:
##Create Time Windows

#Human activity recognition usually uses 2–5 second windows.

#Example: 2-second window

window_size = 400   # if sampling rate ≈ 200 Hz

In [6]:
#Feature Extraction Function
def extract_features(window):

    features = {}

    features['mean_x'] = window['gFx'].mean()
    features['mean_y'] = window['gFy'].mean()
    features['mean_z'] = window['gFz'].mean()

    features['std_x'] = window['gFx'].std()
    features['std_y'] = window['gFy'].std()
    features['std_z'] = window['gFz'].std()

    features['max_x'] = window['gFx'].max()
    features['max_y'] = window['gFy'].max()
    features['max_z'] = window['gFz'].max()

    features['min_x'] = window['gFx'].min()
    features['min_y'] = window['gFy'].min()
    features['min_z'] = window['gFz'].min()

    features['mean_magnitude'] = window['TgF'].mean()
    features['std_magnitude'] = window['TgF'].std()

    return features

In [7]:
#Apply Feature Extraction to Windows
feature_list = []

for start in range(0, len(df), window_size):

    window = df.iloc[start:start+window_size]

    if len(window) == window_size:
        feats = extract_features(window)
        feature_list.append(feats)

features_df = pd.DataFrame(feature_list)

In [8]:
features_df.head()

,mean_x,mean_y,mean_z,std_x,std_y,std_z,max_x,max_y,max_z,min_x,min_y,min_z,mean_magnitude,std_magnitude
0,0.028269,0.35055,0.912256,0.004733,0.00651,0.010465,0.0567,0.3713,0.9518,0.0176,0.3352,0.8727,0.977625,0.009917


In [ ]:
#Each row now represents one time window.

Add Activity Labels (Important for ML)

If collecting activity data like:

walking

sitting

running

Add a label column:

In [9]:
features_df['activity'] = 'sitting'

In [10]:
features_df.to_csv("ml_featuresHAR_sitting.csv", index=False)

In [11]:
df1 = pd.read_csv("/content/drive/MyDrive/VAT datasets/ACCdata/featureactivityfile/ml_featuresHAR_sitting.csv")
df2 = pd.read_csv("/content/drive/MyDrive/VAT datasets/ACCdata/featureactivityfile/ml_featuresHARwalking.csv")

In [12]:
df_combined = pd.concat([df1, df2], axis=0)

In [13]:
df_combined

,mean_x,mean_y,mean_z,std_x,std_y,std_z,max_x,max_y,max_z,min_x,min_y,min_z,mean_magnitude,std_magnitude,activity
0,0.028269,0.350550,0.912256,0.004733,0.006510,0.010465,0.0567,0.3713,0.9518,0.0176,0.3352,0.8727,0.977625,0.009917,sitting
0,0.005560,0.408954,0.895612,0.093915,0.111409,0.123259,0.2511,0.6225,1.3974,-0.2267,0.1300,0.6010,0.995800,0.119780,walking
1,0.011444,0.422417,0.885783,0.094347,0.104916,0.141410,0.2883,0.6069,1.3427,-0.2130,0.1612,0.5013,0.992125,0.136353,walking
2,-0.006186,0.455325,0.868505,0.104366,0.106634,0.125376,0.2756,0.6880,1.2039,-0.2492,0.2101,0.5609,0.992750,0.118579,walking
3,-0.010524,0.489387,0.842858,0.101217,0.117435,0.116456,0.2707,0.7388,1.1824,-0.2199,0.1808,0.5834,0.987275,0.112151,walking


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df_combined.drop('activity', axis=1)
y = df_combined['activity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 1.0
